# 05 Observable factors (B-17)

D-21: automatic factors only. Confidence and model agreement are proxies, not measured correctness.

In [ ]:
import pandas as pd

from humper import categories, data, evaluate, explain, failures

OUT = data.REPO_ROOT / "results"
TAB = OUT / "tables"
scores = pd.read_csv(OUT / "scores.csv")
predictions = pd.read_csv(OUT / "predictions.csv")
measures = pd.read_csv(OUT / "categories.csv").set_index("image_id")

In [ ]:
rows = []
for name, spec in explain.EXPLAINERS.items():
    part = predictions[predictions.model == spec.model].set_index("image_id")
    rows.append(evaluate.spearman_factor(scores, name, "confidence", part.top1_prob))
    rows.append(evaluate.spearman_factor(scores, name, "attention_area", measures.attention_area))
table = pd.concat(rows, ignore_index=True)
table["holm_p"] = evaluate.holm(table.p)
table.to_csv(TAB / "factor_correlations.csv", index=False)
table.round(3)

In [ ]:
wide = predictions.pivot(index="image_id", columns="model", values="top1_idx")
flags = pd.DataFrame({"models_agree": wide.resnet50 == wide.vit_b16})
agreement = failures.category_effects(scores, flags)
agreement.to_csv(TAB / "model_agreement_effect.csv", index=False)
agreement.round(3)